## Imports & Dependencies
Load required Python/PyTorch/Sklearn modules used throughout the notebook.

In [ ]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import DataLoader, TensorDataset, Dataset
import torch.nn as nn
from evaluate import evaluate
import utils as utils
from typing import Callable, Iterator
from tqdm import trange
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import logging
import os
from torch.nn import TransformerEncoder, TransformerEncoderLayer

## Experiment Arguments
Define CLI-style arguments for data/experiment paths and checkpoint restore options.

In [ ]:
import argparse
import logging
parser = argparse.ArgumentParser()
parser.add_argument("--data_dir", default="E:/Desktop/SPEIT302/AAAAAAA/codes")
parser.add_argument("--experiment_dir", default="experiments/base")  
parser.add_argument("--restore_file", default="None")

## Load Dataset & Split
Load `.npz` samples, encode labels, and split into train/val/test subsets.

In [ ]:
import os
import numpy as np
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import DataLoader, Dataset, random_split

data_dir = 'D:/pkl' # path of pkl files
target_labels = ['call', 'meet', 'up', 'down']
X_train, X_val, X_test = [], [], []
y_train, y_val, y_test = [], [], []
label_encoder = LabelEncoder()
label_encoder.fit(target_labels)

for file_name in os.listdir(data_dir):
    if file_name.endswith('.npz'):
        for label in target_labels:
            if label in file_name:
                file_path = os.path.join(data_dir, file_name)
                data = np.load(file_path)
                X = data['X'].astype(np.float32)
                labels = data['labels']
                encoded_label = label_encoder.transform([label])[0]
                
                train_size = int(0.7 * len(X))
                val_size = int(0.1 * len(X))
                test_size = len(X) - train_size - val_size
                
                X_train.append(X[:train_size])
                X_val.append(X[train_size:train_size+val_size])
                X_test.append(X[train_size+val_size:])
                
                y_train.extend([encoded_label] * train_size)
                y_val.extend([encoded_label] * val_size)
                y_test.extend([encoded_label] * test_size)

X_train = np.concatenate(X_train, axis=0)
X_val = np.concatenate(X_val, axis=0)
X_test = np.concatenate(X_test, axis=0)

y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.shape, y_val.shape, y_test.shape)

## Build DataLoaders
Wrap numpy arrays into PyTorch `Dataset`/`DataLoader` objects for training and evaluation.

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        label = self.labels[idx]
        return sample, torch.tensor(label, dtype=torch.long) 

train_dataset = CustomDataset(X_train, y_train)
val_dataset = CustomDataset(X_val, y_val)
test_dataset = CustomDataset(X_test, y_test)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"number of training batches: {len(train_loader)}")
print(f"number of validation batches: {len(val_loader)}")
print(f"number of test batches: {len(test_loader)}")

X_batch, y_batch = next(iter(train_loader))
print(f"X_batch shape: {X_batch.shape}")
print(f"y_batch shape: {y_batch.shape}")

## Training & Testing Routines
Define the training loop, validation checkpointing, early stopping, and final test reporting.

In [ ]:
def train_epoch(
        model: torch.nn.Module,
        optimizer: torch.optim.Optimizer,
        loss_fn: Callable[[torch.Tensor, torch.Tensor], torch.FloatTensor],
        data_iterator: Iterator[tuple[torch.Tensor, torch.Tensor]],
        metrics: dict[str, Callable[[torch.Tensor, torch.Tensor], torch.FloatTensor]],
        params: utils.HyperParams,
        num_steps: int
):
    """
    Train the models on `num_steps` batches/iterations of size `params.batch_size` as one epoch.
    Args:
        * model: (nn.Module) the neural network
        * optimizer: (torch.optim.Optimizer) the optimizer for parameters in the models
        * loss_fn: (Callable) predicted_proba_batch, true_labels_batch -> train_loss
        * data_iterator: (Generator) -> train_batch, true_labels_batch
        * metrics: (dict) metric_name -> (function (Callable) predicted_proba_batch, true_labels_batch -> metric_value)
        * params: (utils.HyperParams) hyperparameters
        * num_steps: (int) number of batches to train for each epoch
    """
    model.train()
    summary: list[dict[str, float]] = []
    loss_avg = utils.RunningAverage()

    t = trange(num_steps)
    for step in t:
        # core pipeline
        train_batch, true_labels_batch = next(data_iterator)
        if params.cuda_index > -1:
            train_batch = train_batch.cuda(device=torch.device(params.cuda_index))
            true_labels_batch = true_labels_batch.cuda(device=torch.device(params.cuda_index))
        predicted_proba_batch: torch.Tensor = model(train_batch, valid_lens=None)
        train_loss = loss_fn(predicted_proba_batch, true_labels_batch)
        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        # evaluate summaries once in a while within one epoch
        if step % params.save_summary_steps == 0:
            predicted_proba_batch = predicted_proba_batch.detach().cpu().numpy()
            true_labels_batch = true_labels_batch.detach().cpu().numpy()
            batch_summary = {metric: metrics[metric](predicted_proba_batch, true_labels_batch) for metric in metrics}
            batch_summary["train_loss"] = train_loss.item()
            summary.append(batch_summary)

        loss_avg.update(train_loss.item())
        t.set_postfix(train_loss="{:05.3f}".format(loss_avg()))

    metrics_mean = {metric: np.mean([batch[metric] for batch in summary]) for metric in summary[0]}
    metrics_string = " ; ".join("{}: {:05.3f}".format(key, value) for key, value in metrics_mean.items())
    logging.info("- Train metrics: " + metrics_string)
    
def test(
        model: torch.nn.Module,
        test_dataloader: DataLoader,
        metrics: dict[str, Callable[[torch.Tensor, torch.Tensor], torch.FloatTensor]],
        params: utils.HyperParams,
        experiment_dir: str
) -> dict[str, float]:
    """
    Test the model on the test set and output confusion matrix, accuracy, precision, recall, and F1 score.
    Args:
        * model: (nn.Module) the neural network
        * test_dataloader: (DataLoader) for test set
        * metrics: (dict) metric_name -> (function (Callable) output_batch, labels_batch -> metric_value)
        * params: (utils.HyperParams) hyperparameters
        * experiment_dir: (str) directory containing config, checkpoints and log
    Returns:
        * metrics: (dict) metric_name -> metric_value on the test set
    """
    # Load the best model weights
    best_model_path = os.path.join(experiment_dir, "best.pth.tar")
    if os.path.isfile(best_model_path):
        logging.info("Loading best model weights from {}".format(best_model_path))
        utils.load_checkpoint(best_model_path, model)
    else:
        logging.error("No best model weights found at {}".format(best_model_path))
        return {}

    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for test_batch, true_labels_batch in test_dataloader:
            if params.cuda_index > -1:
                test_batch = test_batch.cuda(device=torch.device(params.cuda_index))
                true_labels_batch = true_labels_batch.cuda(device=torch.device(params.cuda_index))

            predicted_proba_batch: torch.Tensor = model(test_batch, valid_lens=None)
            predicted_labels_batch = torch.argmax(predicted_proba_batch, dim=1)  

            all_preds.extend(predicted_labels_batch.cpu().numpy())  
            all_labels.extend(true_labels_batch.cpu().numpy())  

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='macro')
    recall = recall_score(all_labels, all_preds, average='macro')
    f1 = f1_score(all_labels, all_preds, average='macro')
    conf_matrix = confusion_matrix(all_labels, all_preds)

    # Log metrics
    logging.info("Test Metrics:")
    logging.info(f"Accuracy: {accuracy:.4f}")
    logging.info(f"Precision: {precision:.4f}")
    logging.info(f"Recall: {recall:.4f}")
    logging.info(f"F1 Score: {f1:.4f}")
    logging.info("Confusion Matrix:")
    logging.info(conf_matrix)

    # Save metrics to a file
    test_metrics = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "confusion_matrix": conf_matrix.tolist()
    }
    test_json_path = os.path.join(experiment_dir, "metrics_test_best_weights.json")
    utils.save_metrics(test_metrics, test_json_path)

    return test_metrics


def train2(
        model: torch.nn.Module,
        optimizer: torch.optim.Optimizer,
        loss_fn: Callable[[torch.Tensor, torch.Tensor], torch.FloatTensor],
        train_dataloader: DataLoader,
        val_dataloader: DataLoader,
        test_dataloader: DataLoader,  
        metrics: dict[str, Callable[[torch.Tensor, torch.Tensor], torch.FloatTensor]],
        params: utils.HyperParams,
        experiment_dir: str
) -> dict[str, float]:
    """
    Train and evaluate the models on `params.num_epochs` epochs, save checkpoints and metrics.
    Args:
        * model: (nn.Module) the neural network
        * optimizer: (torch.optim.Optimizer) the optimizer for parameters in the models
        * loss_fn: (Callable) output_batch, labels_batch -> loss
        * train_dataloader: (TCDataLoader) for training set
        * val_dataloader: (TCDataLoader) for validation set
        * metrics: (dict) metric_name -> (function (Callable) output_batch, labels_batch -> metric_value)
        * params: (utils.Params) hyperparameters
        * experiment_dir: (str) directory containing config, checkpoints and log
    Returns:
        * metrics: (dict) metric_name -> metric_value on the val set of the best epoch
    """
    # reload weights from checkpoint is available
    if args.restore_file and os.path.isfile(os.path.join(args.experiment_dir, args.restore_file + ".pth.tar")):
        restore_path = os.path.join(args.experiment_dir, args.restore_file + ".pth.tar")
        logging.info("Restoring parameters from {}".format(restore_path))
        utils.load_checkpoint(restore_path, model, optimizer)

    best_val_acc = 0.0
    best_metrics: dict[str, float] = {}
    patience = 5  
    no_improve_epochs = 0  

    for epoch in range(params.num_epochs):
        logging.info("Epoch {}/{}".format(epoch + 1, params.num_epochs))

        # train
        num_steps = int((params.train_size + 1) // params.batch_size)
        train_data_iterator = iter(train_dataloader)
        train_epoch(
            model=model,
            optimizer=optimizer,
            loss_fn=loss_fn,
            data_iterator=train_data_iterator,
            metrics=metrics,
            params=params,
            num_steps=num_steps
        )

        # evaluate
        num_steps = int((params.val_size + 1) // params.batch_size)
        val_data_iterator = iter(val_dataloader)
        val_metrics = evaluate(model, loss_fn, val_data_iterator, metrics, params, num_steps)
        val_acc = val_metrics["accuracy"]
        is_best = (val_acc >= best_val_acc)

        # save weights checkpoint
        utils.save_checkpoint(
            state={"epoch": epoch + 1, "state_dict": model.state_dict(), "optim_dict": optimizer.state_dict()},
            is_best=is_best,
            checkpoint_dir=experiment_dir
        )

        # overwrite the best metrics evaluation result if the models is the best by far
        if is_best:
            best_metrics = val_metrics
            logging.info("- Found new best accuracy")
            best_val_acc = val_acc
            best_json_path = os.path.join(experiment_dir, "metrics_val_best_weights.json")
            utils.save_metrics(val_metrics, best_json_path)
            no_improve_epochs = 0  
        else:
            no_improve_epochs += 1  

        if no_improve_epochs >= patience:
            logging.info(f"Early stopping at epoch {epoch + 1} due to no improvement in validation accuracy.")
            break

        # overwrite last metrics evaluation result
        last_json_path = os.path.join(experiment_dir, "metric_val_last_weights.json")
        utils.save_metrics(val_metrics, last_json_path)

    test_metrics = test(
        model=model,
        test_dataloader=test_dataloader,
        metrics=metrics,
        params=params,
        experiment_dir=experiment_dir
    )
    logging.info("Test Metrics: " + str(test_metrics))

    return best_metrics

## Hyperparameters & Logging Setup
Load experiment hyperparameters, set random seeds, and configure logging.

In [ ]:
args, unknown = parser.parse_known_args()
json_path = os.path.join(args.experiment_dir, "params.json")
if not os.path.isfile(json_path):
    raise RuntimeError("Failed to load hyperparameters: no json file found at {}.".format(json_path))
params = utils.HyperParams(json_path)

# bypass cuda index hyperparameter if specified cuda device is not available
if params.cuda_index >= torch.cuda.device_count():
    params.cuda_index = -1

# set random seed for reproducibility
torch.manual_seed(params.random_seed)
if params.cuda_index > -1:
    torch.cuda.manual_seed(params.random_seed)

# set logger
utils.set_logger(os.path.join(args.experiment_dir, "output.log"))

## Model Definition (BiLCNet)
Define the BiLCNet architecture: BiLSTM encoder + Conformer blocks + attention pooling + classifier head.

In [ ]:
class ConformerBlock(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.ffn1 = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
            nn.Dropout(dropout)
        )
        self.conv = nn.Sequential(
            nn.Conv1d(d_model, d_model, kernel_size=3, padding=1, groups=d_model),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.mha = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.ffn2 = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
            nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # Feed forward
        x = x + 0.5 * self.dropout(self.ffn1(self.norm1(x)))
        # Convolution
        x = x + self.dropout(self.conv(self.norm2(x).transpose(1, 2)).transpose(1, 2))
        # Multi-head attention
        attn_out, _ = self.mha(self.norm3(x), self.norm3(x), self.norm3(x))
        x = x + self.dropout(attn_out)
        # Feed forward
        x = x + 0.5 * self.dropout(self.ffn2(x))
        return x

class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.Tanh(),
            nn.Linear(d_model, 1),
            nn.Softmax(dim=1)
        )
        
    def forward(self, x):
        # x: [batch_size, seq_len, d_model]
        weights = self.attention(x)  # [batch_size, seq_len, 1]
        pooled = torch.sum(weights * x, dim=1)  # [batch_size, d_model]
        return pooled

class BiLCNet(nn.Module):
    def __init__(self, input_dim=203, hidden_dim=64, lstm_layers=2, num_classes=4, 
                 num_conformer_layers=2, nhead=8, dropout=0.1):
        super().__init__()
        # LSTM 
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0
        )
        
        # Conformer 
        self.conformer_input_dim = hidden_dim * 2
        self.conformer_layers = nn.ModuleList([
            ConformerBlock(self.conformer_input_dim, nhead, dim_feedforward=128, dropout=dropout)
            for _ in range(num_conformer_layers)
        ])
        
        # Attention Pooling
        self.attention_pooling = AttentionPooling(self.conformer_input_dim)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(self.conformer_input_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x, **kwargs):
        # x: [batch_size, seq_len, input_dim]
        lstm_out, _ = self.lstm(x)  # [batch_size, seq_len, hidden_dim*2]
        
        # Conformer layers
        conformer_out = lstm_out
        for layer in self.conformer_layers:
            conformer_out = layer(conformer_out)
        
        # Attention pooling
        pooled = self.attention_pooling(conformer_out)  # [batch_size, hidden_dim*2]
        
        logits = self.classifier(pooled)
        return logits

## Train a Single Run
Instantiate BiLCNet, move to GPU if available, and start training with the current data split.

In [ ]:
net7 = BiLCNet(
    input_dim=203, hidden_dim=64, lstm_layers=2, num_classes=4, 
    num_conformer_layers=2, nhead=8, dropout=0.1
)
params.train_size = len(train_loader.dataset)  
params.val_size = len(val_loader.dataset)  
# train
if params.cuda_index > -1:
    net7.cuda(device=torch.device(params.cuda_index))
optimizer = torch.optim.AdamW(net7.parameters(), lr=0.001, weight_decay=0.01)
train2(
    model=net7,
    optimizer=optimizer,
    loss_fn=utils.loss_fn,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    test_dataloader=test_loader,
    metrics=utils.metrics,
    params=params,
    experiment_dir=args.experiment_dir
)

## LODO Evaluation
Loop over different `gain` values: hold one gain out for testing and train on the remaining gains.

In [ ]:
data_dir = 'D:/pkl'
target_labels = ['call', 'meet', 'up', 'down']
gains = [66, 68, 70, 72, 74, 76, 78, 80, 82, 84]
label_encoder = LabelEncoder()
label_encoder.fit(target_labels)

class CustomDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        label = self.labels[idx]
        return sample, torch.tensor(label, dtype=torch.long)

def split_file_by_time(data, labels, train_ratio=0.8):
    num_samples = len(data)
    train_size = int(num_samples * train_ratio)
    train_data = data[:train_size]
    train_labels = labels[:train_size]
    val_data = data[train_size:]
    val_labels = labels[train_size:]
    return train_data, train_labels, val_data, val_labels

for test_gain in gains:
    logging.info(f"Testing with gain: {test_gain}")
    X_train, y_train = [], []
    X_val, y_val = [], []
    X_test, y_test = [], []

    for file_name in os.listdir(data_dir):
        if file_name.endswith('.npz'):
            for label in target_labels:
                if label in file_name:
                    file_path = os.path.join(data_dir, file_name)
                    data = np.load(file_path)
                    X = data['X'].astype(np.float32)
                    labels = data['labels']
                    gain = int(file_name.split('_')[-1].split('.')[0])  

                    if gain == test_gain:
                        X_test.append(X)
                        y_test.extend([label_encoder.transform([label])[0]] * len(labels))
                    else:
                        train_data, train_labels, val_data, val_labels = split_file_by_time(X, labels, train_ratio=0.8)
                        X_train.append(train_data)
                        y_train.extend([label_encoder.transform([label])[0]] * len(train_labels))
                        X_val.append(val_data)
                        y_val.extend([label_encoder.transform([label])[0]] * len(val_labels))

    X_train = np.concatenate(X_train, axis=0)
    y_train = np.array(y_train)
    X_val = np.concatenate(X_val, axis=0)
    y_val = np.array(y_val)
    X_test = np.concatenate(X_test, axis=0)
    y_test = np.array(y_test)

    train_dataset = CustomDataset(X_train, y_train)
    val_dataset = CustomDataset(X_val, y_val)
    test_dataset = CustomDataset(X_test, y_test)

    batch_size = 32
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    logging.info(f"number of train batches: {len(train_loader)}")
    logging.info(f"number of val batches: {len(val_loader)}")
    logging.info(f"number of test batches: {len(test_loader)}")
    X_batch, y_batch = next(iter(train_loader))
    logging.info(f"X_batch shape: {X_batch.shape}")
    logging.info(f"y_batch shape: {y_batch.shape}")

    net_trans = BiLCNet(
        input_dim=203, hidden_dim=64, lstm_layers=2, num_classes=4, 
        num_conformer_layers=2, nhead=8, dropout=0.1
    )
    params.train_size = len(train_loader.dataset)  
    params.val_size = len(val_loader.dataset) 
    if params.cuda_index > -1:
        net_trans.cuda(device=torch.device(params.cuda_index))
    optimizer = torch.optim.AdamW(net_trans.parameters(), lr=0.001, weight_decay=0.01)

    train2(
        model=net_trans,
        optimizer=optimizer,
        loss_fn=utils.loss_fn,
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        test_dataloader=test_loader,
        metrics=utils.metrics,
        params=params,
        experiment_dir=args.experiment_dir
    )
    logging.info("\n" + "=" * 50 + "\n")